# Dependencies and Filepath Check

This notebook installs required Python packages (if missing) and validate required files which may be in different locations.

In [ ]:
import sys
import subprocess
import glob
import os

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f"{import_name} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Packages to ensure are available (package-name, import_name)
packages = [
    ("numpy", "numpy"),
    ("Pillow", "PIL"),
    ("tqdm", "tqdm"),
    ("ipywidgets", "ipywidgets"),
    ("pandas", "pandas"),
    ("opencv-python", "cv2"),
    ("scikit-learn", "sklearn"),
    ("timm", "timm"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
]

for pkg, imp in packages:
    install_if_missing(pkg, imp)




In [ ]:
# Filepath check (adjust IMAGE_ROOT if needed)
IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"
all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
path_lookup = {os.path.basename(p): p for p in all_png}
print(f"Found {len(path_lookup):,} images on disk under {IMAGE_ROOT}.")



In [ ]:
# Validation checks for metadata CSV and SwinV2 checkpoint
import os
import glob
from pathlib import Path


METADATA_CSV_PATH = r"..\chest_xray_dataset\CXR8\Data_Entry_2017_v2020.csv"
SWINV2_CKPT = r"..\chest_xray_dataset\swinv2_small_1k_500k.pth"
# Define an array of Path+label tuples and run report_path for each
PATH_CHECKS = [
    (Path(METADATA_CSV_PATH), 'Metadata CSV'),
    (Path(SWINV2_CKPT), 'SwinV2 checkpoint'),
]



def report_path(p, label):
    p_str = str(p)
    exists = os.path.exists(p_str)
    print(f"{label}: {p_str} -> {'FOUND' if exists else 'MISSING'}")
    if not exists:
        # look for nearby matches
        dirname = os.path.dirname(p_str) or '.'
        if 'swinv2' in p_str.lower():
            pattern = os.path.join(dirname, '*swinv2*.pth')
        else:
            pattern = os.path.join(dirname, '*.csv')
        matches = glob.glob(pattern)
        if matches:
            print('  Nearby matches:')
            for m in matches[:10]:
                print('   -', m)
        else:
            print('  No nearby matches found')
    else:
        try:
            size = os.path.getsize(p_str)
            print(f'  size: {size/1024/1024:.2f} MB')
        except Exception:
            pass
        if p_str.lower().endswith('.csv'):
            try:
                import pandas as pd
                df = pd.read_csv(p_str, nrows=5)
                print('  CSV sample:')
                print(df.head().to_string(index=False))
            except Exception as e:
                print('  Could not read CSV sample:', e)



for p, label in PATH_CHECKS:
    report_path(p, label)
